In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:

df = pd.read_csv("/content/drive/MyDrive/Copy of yelp_academic_dataset_review_2.csv")

In [ ]:

import pandas as pd
import fasttext
import gc

language_model = fasttext.load_model("/content/lid.176.ftz")

input_path = (
    "/content/drive/MyDrive/Copy of yelp_academic_dataset_review_2.csv"
)

output_path = (
    "/content/drive/MyDrive/yelp_reviews_with_language.csv"
)

chunk_size = 20_000
first_chunk = True
total_rows = 0

for chunk_number, chunk in enumerate(
    pd.read_csv(input_path, chunksize=chunk_size),
    start=1
):
    texts = (
        chunk["text"]
        .fillna("")
        .astype(str)
        .str.replace("\n", " ", regex=False)
        .str.strip()
        .tolist()
    )

    # FastText can fail on empty strings, so replace them
    texts = [text if text else " " for text in texts]

    predictions, probabilities = language_model.predict(texts, k=1)

    chunk["language"] = [
        labels[0].replace("__label__", "")
        for labels in predictions
    ]

    chunk["language_probability"] = [
        float(scores[0])
        for scores in probabilities
    ]

    chunk.loc[
        chunk["language_probability"] < 0.70,
        "language"
    ] = "unknown"

    chunk.to_csv(
        output_path,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False
    total_rows += len(chunk)

    print(
        f"Finished chunk {chunk_number}; "
        f"processed {total_rows:,} rows"
    )

    del chunk, texts, predictions, probabilities
    gc.collect()

print("Language detection completed.")

Finished chunk 1; processed 20,000 rows
Finished chunk 2; processed 40,000 rows
Finished chunk 3; processed 60,000 rows
Finished chunk 4; processed 80,000 rows
Finished chunk 5; processed 100,000 rows
Finished chunk 6; processed 120,000 rows
Finished chunk 7; processed 140,000 rows
Finished chunk 8; processed 160,000 rows
Finished chunk 9; processed 180,000 rows
Finished chunk 10; processed 200,000 rows
Finished chunk 11; processed 220,000 rows
Finished chunk 12; processed 240,000 rows
Finished chunk 13; processed 260,000 rows
Finished chunk 14; processed 280,000 rows
Finished chunk 15; processed 300,000 rows
Finished chunk 16; processed 320,000 rows
Finished chunk 17; processed 340,000 rows
Finished chunk 18; processed 360,000 rows
Finished chunk 19; processed 380,000 rows
Finished chunk 20; processed 400,000 rows
Finished chunk 21; processed 420,000 rows
Finished chunk 22; processed 440,000 rows
Finished chunk 23; processed 460,000 rows
Finished chunk 24; processed 480,000 rows
Finis

In [ ]:
import pandas as pd

output_path = (
    "/content/drive/MyDrive/yelp_reviews_with_language.csv"
)

language_counts = {}

for chunk in pd.read_csv(
    output_path,
    usecols=["language"],
    chunksize=100_000
):
    counts = chunk["language"].value_counts()

    for language, count in counts.items():
        language_counts[language] = (
            language_counts.get(language, 0) + count
        )

language_counts = pd.Series(language_counts).sort_values(
    ascending=False
)

print(language_counts.head(20))

en         6970895
unknown      14571
es            2363
de             785
fr             619
zh             262
pt             192
ja             187
it             148
sv             116
nl              56
da              25
tr              20
vi              10
hu               5
fy               4
pl               4
eo               3
no               3
ceb              3
dtype: int64


In [ ]:
import pandas as pd
import gc

input_path = (
    "/content/drive/MyDrive/yelp_reviews_with_language.csv"
)

output_path = (
    "/content/drive/MyDrive/yelp_reviews_english_only.csv"
)

chunk_size = 100_000
first_chunk = True
total_english_rows = 0

for chunk_number, chunk in enumerate(
    pd.read_csv(input_path, chunksize=chunk_size),
    start=1
):
    english_chunk = chunk[chunk["language"] == "en"].copy()

    if not english_chunk.empty:
        english_chunk.to_csv(
            output_path,
            mode="w" if first_chunk else "a",
            header=first_chunk,
            index=False
        )

        first_chunk = False
        total_english_rows += len(english_chunk)

    print(
        f"Finished chunk {chunk_number}; "
        f"saved {total_english_rows:,} English reviews"
    )

    del chunk, english_chunk
    gc.collect()

print("English-only CSV created.")
print("Total English reviews:", total_english_rows)

Finished chunk 1; saved 99,702 English reviews
Finished chunk 2; saved 199,375 English reviews
Finished chunk 3; saved 299,066 English reviews
Finished chunk 4; saved 398,794 English reviews
Finished chunk 5; saved 498,550 English reviews
Finished chunk 6; saved 598,287 English reviews
Finished chunk 7; saved 698,117 English reviews
Finished chunk 8; saved 797,776 English reviews
Finished chunk 9; saved 897,470 English reviews
Finished chunk 10; saved 997,125 English reviews
Finished chunk 11; saved 1,096,817 English reviews
Finished chunk 12; saved 1,196,561 English reviews
Finished chunk 13; saved 1,296,324 English reviews
Finished chunk 14; saved 1,396,133 English reviews
Finished chunk 15; saved 1,495,821 English reviews
Finished chunk 16; saved 1,595,494 English reviews
Finished chunk 17; saved 1,695,152 English reviews
Finished chunk 18; saved 1,794,832 English reviews
Finished chunk 19; saved 1,894,568 English reviews
Finished chunk 20; saved 1,994,338 English reviews
Finished c

In [ ]:
import pandas as pd
df1=pd.read_csv( "/content/drive/MyDrive/yelp_reviews_english_only.csv")
df1.head()

,review_id,user_id,business_id,stars,useful,funny,cool,text,date,language,language_probability
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11,en,0.989625
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5.0,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18,en,0.968923
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3.0,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30,en,0.922908
3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5.0,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03,en,0.910928
4,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4.0,1,0,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15,en,0.975548


In [ ]:
import pandas as pd
import numpy as np
import re
import os
import gc
import time

start_time = time.time()

# ---------------------------------------------------------
# File paths
# ---------------------------------------------------------
reviews_path = (
    "/content/drive/MyDrive/yelp_reviews_english_only.csv"
)

business_path = (
    "/content/drive/MyDrive/yelp_academic_dataset_business.csv"
)

output_path = (
    "/content/drive/MyDrive/yelp_reviews_clean.csv"
)

# Delete an existing output file so data are not duplicated
if os.path.exists(output_path):
    os.remove(output_path)



# 1. Load and prepare the smaller business dataset

business_clean = pd.read_csv(
    business_path,
    usecols=["business_id", "name"],
    dtype={
        "business_id": "string",
        "name": "string"
    }
)

business_clean = (
    business_clean
    .dropna(subset=["business_id", "name"])
    .drop_duplicates(subset=["business_id"], keep="first")
    .rename(columns={"name": "business_name"})
)

business_clean["business_name"] = (
    business_clean["business_name"]
    .str.strip()
)

business_clean = business_clean[
    business_clean["business_name"].ne("")
].copy()


# 2. Define the columns needed from the review dataset

review_columns = [
    "review_id",
    "user_id",
    "business_id",
    "stars",
    "date",
    "text"
]

review_dtypes = {
    "review_id": "string",
    "user_id": "string",
    "business_id": "string",
    "stars": "float32",
    "text": "string"
}



# 3. Process the review dataset in chunks

chunk_size = 100_000
first_chunk = True
total_input_rows = 0
total_output_rows = 0

reader = pd.read_csv(
    reviews_path,
    usecols=review_columns,
    dtype=review_dtypes,
    chunksize=chunk_size,
    engine="c"
)

for chunk_number, reviews_clean in enumerate(reader, start=1):

    input_rows = len(reviews_clean)
    total_input_rows += input_rows

    # Remove missing required values

    reviews_clean = reviews_clean.dropna(
        subset=["business_id", "text", "stars", "date"]
    )

    # Clean text with vectorized string operations

    text = reviews_clean["text"]

    # Unicode normalization
    text = text.str.normalize("NFKC")

    # Standardize quotation marks, dashes, and spaces
    text = text.str.replace("’", "'", regex=False)
    text = text.str.replace("‘", "'", regex=False)
    text = text.str.replace("“", '"', regex=False)
    text = text.str.replace("”", '"', regex=False)
    text = text.str.replace("–", "-", regex=False)
    text = text.str.replace("—", "-", regex=False)
    text = text.str.replace("\u00A0", " ", regex=False)

    # Remove URLs
    text = text.str.replace(
        r"https?://\S+|www\.\S+",
        " ",
        regex=True
    )

    # Replace line breaks and tabs
    text = text.str.replace(
        r"[\r\n\t]+",
        " ",
        regex=True
    )

    # Remove control characters
    text = text.str.replace(
        r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]",
        "",
        regex=True
    )

    # Standardize repeated whitespace
    text = text.str.replace(
        r"\s+",
        " ",
        regex=True
    ).str.strip()

    reviews_clean["text"] = text

    # Remove empty reviews after cleaning
    reviews_clean = reviews_clean[
        reviews_clean["text"].ne("")
    ].copy()

    # Process star ratings

    reviews_clean = reviews_clean[
        reviews_clean["stars"].between(1, 5)
    ].copy()

    reviews_clean["sentiment"] = np.select(
        [
            reviews_clean["stars"] > 3,
            reviews_clean["stars"] == 3,
            reviews_clean["stars"] < 3
        ],
        [
            "positive",
            "neutral",
            "negative"
        ],
        default="unknown"
    )

    # Process dates

    reviews_clean["date"] = pd.to_datetime(
        reviews_clean["date"],
        errors="coerce"
    )

    reviews_clean = reviews_clean.dropna(subset=["date"])

    # Produces dates such as 2018-07-07
    reviews_clean["date"] = (
        reviews_clean["date"]
        .dt.strftime("%Y-%m-%d")
    )


    # Remove duplicate review IDs within the chunk

    reviews_clean = reviews_clean.drop_duplicates(
        subset=["review_id"],
        keep="first"
    )

    # Merge with the business dataset

    clean_chunk = reviews_clean.merge(
        business_clean,
        on="business_id",
        how="inner",
        validate="many_to_one",
        sort=False
    )

    # Arrange final columns

    clean_chunk = clean_chunk[
        [
            "review_id",
            "user_id",
            "business_id",
            "business_name",
            "stars",
            "sentiment",
            "date",
            "text"
        ]
    ]


    # Append the cleaned chunk to the output CSV

    clean_chunk.to_csv(
        output_path,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False,
        encoding="utf-8"
    )

    first_chunk = False
    total_output_rows += len(clean_chunk)

    print(
        f"Chunk {chunk_number}: "
        f"read {input_rows:,}, "
        f"saved {len(clean_chunk):,}, "
        f"total saved {total_output_rows:,}"
    )

    del reviews_clean, clean_chunk, text
    gc.collect()


# 4. Display final results

elapsed_minutes = (time.time() - start_time) / 60

print("\nPreprocessing completed.")
print("Input rows processed:", f"{total_input_rows:,}")
print("Clean rows saved:", f"{total_output_rows:,}")
print("Output path:", output_path)
print("Elapsed time:", f"{elapsed_minutes:.2f} minutes")

Chunk 1: read 100,000, saved 100,000, total saved 100,000
Chunk 2: read 100,000, saved 100,000, total saved 200,000
Chunk 3: read 100,000, saved 100,000, total saved 300,000
Chunk 4: read 100,000, saved 100,000, total saved 400,000
Chunk 5: read 100,000, saved 100,000, total saved 500,000
Chunk 6: read 100,000, saved 100,000, total saved 600,000
Chunk 7: read 100,000, saved 100,000, total saved 700,000
Chunk 8: read 100,000, saved 100,000, total saved 800,000
Chunk 9: read 100,000, saved 100,000, total saved 900,000
Chunk 10: read 100,000, saved 100,000, total saved 1,000,000
Chunk 11: read 100,000, saved 100,000, total saved 1,100,000
Chunk 12: read 100,000, saved 100,000, total saved 1,200,000
Chunk 13: read 100,000, saved 100,000, total saved 1,300,000
Chunk 14: read 100,000, saved 100,000, total saved 1,400,000
Chunk 15: read 100,000, saved 100,000, total saved 1,500,000
Chunk 16: read 100,000, saved 100,000, total saved 1,600,000
Chunk 17: read 100,000, saved 100,000, total saved 

In [3]:
import pandas as pd
import os
import gc

reviews_path = "/content/drive/MyDrive/yelp_reviews_clean.csv"
business_path = "/content/drive/MyDrive/yelp_academic_dataset_business.csv"
output_path = "/content/drive/MyDrive/yelp_reviews_clean_CA.csv"

# 1. Get California business IDs

business_df = pd.read_csv(
    business_path,
    usecols=["business_id", "state"],
    dtype={"business_id": "string", "state": "string"}
)

ca_business_ids = set(
    business_df.loc[
        business_df["state"].str.upper() == "CA",
        "business_id"
    ].dropna()
)

print("Number of California businesses:", len(ca_business_ids))

# 2. Create output file with header first

header_df = pd.read_csv(reviews_path, nrows=0)
header_df.to_csv(output_path, index=False, encoding="utf-8")

# 3. Filter reviews in chunks and append rows
chunk_size = 100_000
total_rows_read = 0
total_rows_saved = 0

for chunk_number, chunk in enumerate(
    pd.read_csv(
        reviews_path,
        chunksize=chunk_size,
        dtype={"business_id": "string"}
    ),
    start=1
):
    total_rows_read += len(chunk)

    ca_chunk = chunk[
        chunk["business_id"].isin(ca_business_ids)
    ].copy()

    if not ca_chunk.empty:
        ca_chunk.to_csv(
            output_path,
            mode="a",
            header=False,
            index=False,
            encoding="utf-8"
        )

        total_rows_saved += len(ca_chunk)

    print(
        f"Chunk {chunk_number}: "
        f"read {len(chunk):,}, "
        f"saved {len(ca_chunk):,}, "
        f"total saved {total_rows_saved:,}"
    )

    del chunk, ca_chunk
    gc.collect()

print("\nFiltering completed.")
print("Total rows read:", f"{total_rows_read:,}")
print("Total California review rows saved:", f"{total_rows_saved:,}")
print("Output path:", output_path)

Number of California businesses: 5203
Chunk 1: read 100,000, saved 4,939, total saved 4,939
Chunk 2: read 100,000, saved 4,957, total saved 9,896
Chunk 3: read 100,000, saved 4,895, total saved 14,791
Chunk 4: read 100,000, saved 4,550, total saved 19,341
Chunk 5: read 100,000, saved 4,585, total saved 23,926
Chunk 6: read 100,000, saved 4,704, total saved 28,630
Chunk 7: read 100,000, saved 4,873, total saved 33,503
Chunk 8: read 100,000, saved 4,753, total saved 38,256
Chunk 9: read 100,000, saved 4,750, total saved 43,006
Chunk 10: read 100,000, saved 4,956, total saved 47,962
Chunk 11: read 100,000, saved 4,609, total saved 52,571
Chunk 12: read 100,000, saved 4,257, total saved 56,828
Chunk 13: read 100,000, saved 4,508, total saved 61,336
Chunk 14: read 100,000, saved 4,547, total saved 65,883
Chunk 15: read 100,000, saved 5,332, total saved 71,215
Chunk 16: read 100,000, saved 5,575, total saved 76,790
Chunk 17: read 100,000, saved 5,720, total saved 82,510
Chunk 18: read 100,00